In [1]:
import tensorflow as tf
import keras
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping


2025-10-19 19:10:33.913936: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760915433.932191 3917201 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760915433.937709 3917201 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760915433.952058 3917201 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760915433.952082 3917201 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760915433.952085 3917201 computation_placer.cc:177] computation placer alr

In [2]:
train_image_folder = '/raid/mpsych/OMAMA/DATA/data/train'
train_npz_folder = '/raid/mpsych/OMAMA/DATA/data/2d_resized_512/images'

In [3]:
# Get lists of image and npz files with their labels
# PNG files are synthetic (label 0), NPZ files are original (label 1)
png_files = [os.path.join(train_image_folder, f) for f in os.listdir(train_image_folder) if f.endswith('.png')]
npz_files = [os.path.join(train_npz_folder, f) for f in os.listdir(train_npz_folder) if f.endswith('.npz')]

# Combine all files with their labels
all_files = []
for png_file in png_files:
    all_files.append((png_file, 'png', 0))  # 0 = synthetic
for npz_file in npz_files:
    all_files.append((npz_file, 'npz', 1))  # 1 = original

print(f"Total files: {len(all_files)}")
print(f"PNG files (synthetic): {len(png_files)}")
print(f"NPZ files (original): {len(npz_files)}")


Total files: 263567
PNG files (synthetic): 99999
NPZ files (original): 163568


In [ ]:
# Check file counts
print(f"PNG files: {len(png_files)}")
print(f"NPZ files: {len(npz_files)}")
print(f"Total files: {len(all_files)}")

PNG files: 99999
NPZ files: 163568
Total files: 263567


In [ ]:
len(npz_files)

163568

In [ ]:
# Limit files to 50000 for faster training
all_files = all_files[:50000]
print(f"Using {len(all_files)} files for training")

Using 50000 files for training


In [ ]:
# Split dataset into train, validation, and test sets
train_files, test_files = train_test_split(all_files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(test_files, test_size=0.5, random_state=42)

print(f"Train files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")
print(f"Test files: {len(test_files)}")

Train files: 35000
Validation files: 7500
Test files: 7500


In [8]:
# DIAGNOSTIC: Check data ranges to understand normalization needs
print("Checking data ranges...\n")

# Check PNG file
sample_png = cv2.imread(png_files[0], cv2.IMREAD_GRAYSCALE)
print(f"PNG file: {png_files[0]}")
print(f"  - dtype: {sample_png.dtype}")
print(f"  - shape: {sample_png.shape}")
print(f"  - min: {sample_png.min()}, max: {sample_png.max()}")
print(f"  - mean: {sample_png.mean():.2f}\n")

# Check NPZ file
with np.load(npz_files[0], allow_pickle=True) as data:
    sample_npz = data['data']
print(f"NPZ file: {npz_files[0]}")
print(f"  - dtype: {sample_npz.dtype}")
print(f"  - shape: {sample_npz.shape}")
print(f"  - min: {sample_npz.min()}, max: {sample_npz.max()}")
print(f"  - mean: {sample_npz.mean():.2f}\n")

# Check a few more samples to see if ranges are consistent
print("Checking 5 more samples of each type...")
png_ranges = []
npz_ranges = []
for i in range(1, 6):
    png = cv2.imread(png_files[i], cv2.IMREAD_GRAYSCALE)
    png_ranges.append((png.min(), png.max()))
    
    with np.load(npz_files[i], allow_pickle=True) as data:
        npz = data['data']
    npz_ranges.append((npz.min(), npz.max()))

print(f"PNG ranges: {png_ranges}")
print(f"NPZ ranges: {npz_ranges}")


Checking data ranges...

PNG file: /raid/mpsych/OMAMA/DATA/data/train/sample_64145.png
  - dtype: uint8
  - shape: (640, 512)
  - min: 0, max: 255
  - mean: 32.56

NPZ file: /raid/mpsych/OMAMA/DATA/data/2d_resized_512/images/267934773880390363402441981338430720492.npz
  - dtype: uint16
  - shape: (512, 512)
  - min: 2, max: 3940
  - mean: 809.42

Checking 5 more samples of each type...
PNG ranges: [(0, 255), (0, 255), (0, 255), (0, 255), (0, 255)]
NPZ ranges: [(0, 3506), (0, 3606), (271, 2915), (686, 3301), (54, 3911)]


In [ ]:
# Image dimensions and batch size
img_height = 512
img_width = 512
batch_size = 32

In [10]:
# Function to apply consistent preprocessing to both PNG and NPZ
def preprocess_image(image, is_npz=False):
    """
    Apply consistent preprocessing to make PNG and NPZ files look identical
    This removes format-specific characteristics
    """
    image = image.astype(np.float32)
    
    if is_npz:
        # For NPZ files: apply window/level first, then normalize
        window_center = 400
        window_width = 1200
        window_min = window_center - window_width / 2
        window_max = window_center + window_width / 2
        
        # Apply window/level
        image = np.clip(image, window_min, window_max)
        image = (image - window_min) / (window_max - window_min)
    else:
        # For PNG files: just normalize to 0-1
        image = image / 255.0
    
    # Convert to uint8 for OpenCV operations
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Apply additional preprocessing to make them identical
    # 1. Apply Gaussian blur to reduce noise differences
    image_uint8 = cv2.GaussianBlur(image_uint8, (3, 3), 0)
    
    # 2. Apply histogram equalization to normalize contrast
    image_uint8 = cv2.equalizeHist(image_uint8)
    
    # 3. Apply slight noise reduction
    image_uint8 = cv2.bilateralFilter(image_uint8, 9, 75, 75)
    
    # Convert back to float32 and normalize to 0-1
    image = image_uint8.astype(np.float32) / 255.0
    
    return image

# Function to normalize PNG images
def normalize_png(image):
    return preprocess_image(image, is_npz=False)

# Function to normalize NPZ data
def normalize_npz(npz_data):
    return preprocess_image(npz_data, is_npz=True)

In [ ]:
def data_generator(file_list, batch_size, img_height, img_width):
    # This generator creates batches of images with proper labels
    total_files = len(file_list)
    indices = np.arange(total_files)
    np.random.shuffle(indices)

    while True:
        for i in range(0, total_files, batch_size):
            batch_indices = indices[i:i + batch_size]
            batch_images = []
            batch_labels = []

            for idx in batch_indices:
                file_path, file_type, label = file_list[idx]

                if file_type == 'png':
                    # Load and process PNG file (synthetic)
                    image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (img_width, img_height))
                    image = normalize_png(image)  # PNG is already contrast-adjusted
                    
                elif file_type == 'npz':
                    # Load and process NPZ file (original)
                    with np.load(file_path, allow_pickle=True) as data:
                        image = data['data']
                    image = cv2.resize(image, (img_width, img_height))
                    image = normalize_npz(image)  # Apply window/level to raw NPZ data

                # Add channel dimension
                image = np.expand_dims(image, axis=-1)
                batch_images.append(image)

                # Create one-hot encoded label
                if label == 0:  # synthetic
                    batch_labels.append([1, 0])
                else:  # original
                    batch_labels.append([0, 1])

            # Convert to numpy arrays
            batch_images = np.array(batch_images)
            batch_labels = np.array(batch_labels)
            
            yield (batch_images, batch_labels)




In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, min_delta=0.001, mode='min')

In [ ]:
NUMBER_OF_CLASSES = 2

In [14]:
# Create data generators
train_generator = data_generator(train_files, batch_size, img_height, img_width)
val_generator = data_generator(val_files, batch_size, img_height, img_width)
test_generator = data_generator(test_files, batch_size, img_height, img_width)

print("Data generators created successfully")


Data generators created successfully


In [15]:
# Create the model
model = keras.models.Sequential()

# Convolutional layers
model.add(keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', 
                             input_shape=(img_height, img_width, 1)))
model.add(keras.layers.Conv2D(64, (3, 3), activation='relu'))
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))
model.add(keras.layers.Dropout(0.25))

# Flatten and dense layers
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(128, activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(2, activation='softmax'))  # 2 classes: synthetic vs original

print("Model created successfully")
model.summary()

/home/a.kanamarlapudi001/miniconda3/envs/tf_gpu_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1760915438.782243 3917201 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38380 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:07:00.0, compute capability: 8.0


Model created successfully


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 510, 510, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 508, 508, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 254, 254, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 254, 254, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4129024)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │   528,515,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 528,534,274 (1.97 GB)

 Trainable params: 528,534,274 (1.97 GB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer="adadelta",
              metrics=['accuracy'])

In [ ]:
# Train the model
print("Starting training...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_files) // batch_size,
    epochs=5,  # Increased epochs since we have proper data now
    validation_data=val_generator,
    validation_steps=len(val_files) // batch_size,
    verbose=1,
    callbacks=[early_stopping]
)

# Evaluate the model on test set
print("Evaluating on test set...")
test_loss, test_accuracy = model.evaluate(test_generator, steps=len(test_files) // batch_size)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Print training history
print("\nTraining completed!")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

Starting training...
Epoch 1/5


I0000 00:00:1760915442.373037 3917259 service.cc:152] XLA service 0x7f978800da40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760915442.373076 3917259 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2025-10-19 19:10:42.401156: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1760915442.551126 3917259 cuda_dnn.cc:529] Loaded cuDNN version 90300
Could not load symbol cuFuncGetName. Error: /usr/lib/x86_64-linux-gnu/libcuda.so.1: undefined symbol: cuFuncGetName
2025-10-19 19:10:50.184565: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng23{k2=6,k13=0,k14=2,k18=1,k23=0} for conv %cudnn-conv-bw-filter.2 = (f32[32,1,3,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1,512,512]{3,2,1,0} %bitcast.7045, f32[32,32,510,510]{3,2,1,0} %bitcast.7227), window={s

   2/1093 ━━━━━━━━━━━━━━━━━━━━ 1:48 100ms/step - accuracy: 0.6016 - loss: 0.6387 

I0000 00:00:1760915457.150825 3917259 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 269/1093 ━━━━━━━━━━━━━━━━━━━━ 6:17 459ms/step - accuracy: 0.9878 - loss: 0.0571

In [ ]:
# Test the model on a PNG file (should predict synthetic)
test_png_file = '/raid/mpsych/OMAMA/DATA/data/train/sample_40069.png'
test_image = cv2.imread(test_png_file, cv2.IMREAD_GRAYSCALE)
test_image = cv2.resize(test_image, (img_width, img_height))
test_image = normalize_png(test_image)  # FIXED: Use normalize_png
test_image = np.expand_dims(test_image, axis=-1)  # Add channel dimension
test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension

predictions = model.predict(test_image)
print(f"Testing PNG file: {test_png_file}")
print(f"Predictions: {predictions}")
predicted_class = np.argmax(predictions)
class_name = "Synthetic" if predicted_class == 0 else "Original"
print(f"Predicted Class: {class_name} (class {predicted_class})")

In [ ]:
# Test the model on another PNG file
test_png_file2 = '/raid/mpsych/OMAMA/DATA/data/train/sample_10446.png'
test_image = cv2.imread(test_png_file2, cv2.IMREAD_GRAYSCALE)
test_image = cv2.resize(test_image, (img_width, img_height))
test_image = normalize_png(test_image)  # FIXED: Use normalize_png
test_image = np.expand_dims(test_image, axis=-1)  # Add channel dimension
test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension

predictions = model.predict(test_image)
print(f"Testing PNG file: {test_png_file2}")
print(f"Predictions: {predictions}")
predicted_class = np.argmax(predictions)
class_name = "Synthetic" if predicted_class == 0 else "Original"
print(f"Predicted Class: {class_name} (class {predicted_class})")

In [ ]:
# Test the preprocessing to make sure both PNG and NPZ have similar characteristics
print("Testing preprocessing functions...")

# Test PNG preprocessing
sample_png = cv2.imread(png_files[0], cv2.IMREAD_GRAYSCALE)
sample_png = cv2.resize(sample_png, (img_width, img_height))
sample_png_normalized = normalize_png(sample_png)

print(f"PNG original range: {sample_png.min()} - {sample_png.max()}")
print(f"PNG normalized range: {sample_png_normalized.min():.3f} - {sample_png_normalized.max():.3f}")
print(f"PNG normalized mean: {sample_png_normalized.mean():.3f}")

# Test NPZ preprocessing
with np.load(npz_files[0], allow_pickle=True) as data:
    sample_npz = data['data']
sample_npz = cv2.resize(sample_npz, (img_width, img_height))
sample_npz_normalized = normalize_npz(sample_npz)

print(f"\nNPZ original range: {sample_npz.min()} - {sample_npz.max()}")
print(f"NPZ normalized range: {sample_npz_normalized.min():.3f} - {sample_npz_normalized.max():.3f}")
print(f"NPZ normalized mean: {sample_npz_normalized.mean():.3f}")

print("\nBoth files should now have similar ranges (0.0 - 1.0) and similar characteristics!")


In [ ]:
# Test the model on an NPZ file (should predict original)
test_npz_file = '/raid/mpsych/OMAMA/DATA/data/2d_resized_512/images/100220136299296817993264225430810813957.npz'

with np.load(test_npz_file, allow_pickle=True) as data:
    test_npz = data['data']

# Process the NPZ data exactly like in training
test_npz = cv2.resize(test_npz, (img_width, img_height))
test_npz = normalize_npz(test_npz)  # Use our normalization function
test_npz = np.expand_dims(test_npz, axis=-1)  # Add channel dimension
test_npz = np.expand_dims(test_npz, axis=0)  # Add batch dimension

# Make prediction
predictions = model.predict(test_npz)
print(f"Testing NPZ file: {test_npz_file}")
print(f"Predictions: {predictions}")
predicted_class = np.argmax(predictions)
class_name = "Synthetic" if predicted_class == 0 else "Original"
print(f"Predicted Class: {class_name} (class {predicted_class})")
print(f"Confidence: {np.max(predictions):.4f}")


## Full Test Set Evaluation


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Generate predictions for the entire test set
print("Evaluating on test set...")
test_predictions = []
test_labels = []

for file_path, file_type, label in test_files:
    if file_type == 'png':
        image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (img_width, img_height))
        image = normalize_png(image)
    else:
        with np.load(file_path, allow_pickle=True) as data:
            image = data['data']
        image = cv2.resize(image, (img_width, img_height))
        image = normalize_npz(image)
    
    image = np.expand_dims(image, axis=-1)
    image = np.expand_dims(image, axis=0)
    
    pred = model.predict(image, verbose=0)
    test_predictions.append(np.argmax(pred))
    test_labels.append(label)

# Convert to arrays
test_predictions = np.array(test_predictions)
test_labels = np.array(test_labels)

# Calculate accuracy
test_accuracy = np.mean(test_predictions == test_labels)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Confusion Matrix
cm = confusion_matrix(test_labels, test_predictions)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='RdYlBu_r')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.colorbar()

classes = ['Synthetic', 'Original']
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, fontsize=12)
plt.yticks(tick_marks, classes, fontsize=12)

# Add text annotations
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, f'{cm[i, j]}\n({cm[i, j]/len(test_labels)*100:.1f}%)',
             ha='center', va='center', fontsize=12, fontweight='bold',
             color='white' if cm[i, j] > thresh else 'black')

plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(test_labels, test_predictions, target_names=classes))


In [ ]:
# 3 epochs
# test_file = '/raid/mpsych/OMAMA/DATA/data/train/sample_40069.png'
# test_image = cv2.imread(test_file, cv2.IMREAD_GRAYSCALE)
# test_image = cv2.resize(test_image, (img_width, img_height))
# test_image = np.expand_dims(test_image, axis=-1)
# test_image = test_image / 255.0
# test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension

# predictions = model.predict(test_image)
# print("Predictions:", predictions)
# predicted_class = np.argmax(predictions)
# print("Predicted Class:", predicted_class)